# Olist E-commerce SQL Business Analysis

## Project Overview

This project uses SQL to analyze the Brazilian Olist E-commerce dataset.  
The goal is to answer business questions related to revenue, sellers, customers, product categories, payments, reviews, and delivery performance.

This project demonstrates:
- SQL joins
- Aggregations
- Grouping and ordering
- Date functions
- Business insight generation
- SQL analysis inside Jupyter Notebook

## 1. Import Libraries

In [1]:
import sqlite3
import pandas as pd
import os

## 2. Load CSV Files and Create SQLite Database

In [2]:
# Update this path if your files are in a different folder
DATA_PATH = "data/Raw/archive (11)"

# If the path above does not exist, try alternative common paths
if not os.path.exists(DATA_PATH):
    if os.path.exists("data/Raw"):
        DATA_PATH = "data/Raw"
    elif os.path.exists("data/raw"):
        DATA_PATH = "data/raw"

print("Using data path:", DATA_PATH)

files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Create SQLite database
conn = sqlite3.connect("olist_sql_project.db")

# Load each CSV into SQLite
for table_name, file_name in files.items():
    file_path = os.path.join(DATA_PATH, file_name)
    df = pd.read_csv(file_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"{table_name}: {df.shape}")

print("Database created successfully.")

Using data path: data/Raw/archive (11)
customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
products: (32951, 9)
order_reviews: (99224, 7)
order_payments: (103886, 5)
sellers: (3095, 4)
category_translation: (71, 2)
Database created successfully.


## 3. Check Available Tables

In [3]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql(query, conn)

,name
0,customers
1,orders
2,order_items
3,products
4,order_reviews
5,order_payments
6,sellers
7,category_translation


## 4. Helper Function to Run SQL Queries

In [4]:
def run_query(query):
    return pd.read_sql(query, conn)

## Business Question 1: Top 10 Sellers by Revenue

**Question:** Which sellers generated the highest revenue?

In [5]:
query = """
SELECT
    oi.seller_id,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
GROUP BY oi.seller_id
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

,seller_id,total_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05
2,4a3ca9315b744ce9f8e9374361493884,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55
9,955fee9216a65b617aa5c0531780ce60,135171.70


### Insight

The top sellers generated the highest revenue, suggesting that a small group of sellers contributes significantly to overall sales performance.

## Business Question 2: Top 10 Product Categories by Revenue

**Question:** Which product categories generated the highest revenue?

In [6]:
query = """
SELECT
    p.product_category_name,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

,product_category_name,total_revenue
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32
5,moveis_decoracao,729762.49
6,cool_stuff,635290.85
7,utilidades_domesticas,632248.66
8,automotivo,592720.11
9,ferramentas_jardim,485256.46


### Insight

The highest-revenue product categories show where customer demand is strongest and where Olist may prioritize marketing and inventory planning.

## Business Question 3: Top 10 States by Number of Orders

**Question:** Which customer states generated the highest number of orders?

In [7]:
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC
LIMIT 10;
"""

run_query(query)

,customer_state,total_orders
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


### Insight

The states with the highest order volume represent Olist's strongest customer markets and may be important for logistics and customer acquisition.

## Business Question 4: Top 10 States by Revenue

**Question:** Which customer states generated the highest revenue?

In [8]:
query = """
SELECT
    c.customer_state,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

,customer_state,total_revenue
0,SP,5202955.05
1,RJ,1824092.67
2,MG,1585308.03
3,RS,750304.02
4,PR,683083.76
5,SC,520553.34
6,BA,511349.99
7,DF,302603.94
8,GO,294591.95
9,ES,275037.31


### Insight

The highest-revenue states are key markets where Olist can focus sales strategies and operational resources.

## Business Question 5: Payment Method Distribution

**Question:** Which payment methods are used most frequently?

In [9]:
query = """
SELECT
    payment_type,
    COUNT(*) AS payment_count
FROM order_payments
GROUP BY payment_type
ORDER BY payment_count DESC;
"""

run_query(query)

,payment_type,payment_count
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3


### Insight

The most frequently used payment method reflects customer payment preferences and can help Olist optimize the checkout experience.

## Business Question 6: Revenue by Payment Method

**Question:** Which payment methods generated the highest payment value?

In [10]:
query = """
SELECT
    payment_type,
    ROUND(SUM(payment_value), 2) AS total_payment_value
FROM order_payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;
"""

run_query(query)

,payment_type,total_payment_value
0,credit_card,12542084.19
1,boleto,2869361.27
2,voucher,379436.87
3,debit_card,217989.79
4,not_defined,0.00


### Insight

Payment methods with the highest total value show which payment channels are most important for revenue generation.

## Business Question 7: Customer Review Score Distribution

**Question:** How are customer review scores distributed?

In [11]:
query = """
SELECT
    review_score,
    COUNT(*) AS review_count
FROM order_reviews
GROUP BY review_score
ORDER BY review_score;
"""

run_query(query)

,review_score,review_count
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


### Insight

The distribution of review scores helps evaluate overall customer satisfaction and identify whether most customers had positive or negative experiences.

## Business Question 8: Average Review Score by Product Category

**Question:** Which product categories have the highest average review score?

In [12]:
query = """
SELECT
    p.product_category_name,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM order_reviews r
JOIN orders o
    ON r.order_id = o.order_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY avg_review_score DESC
LIMIT 10;
"""

run_query(query)

,product_category_name,avg_review_score
0,cds_dvds_musicais,4.64
1,fashion_roupa_infanto_juvenil,4.50
2,livros_interesse_geral,4.45
3,construcao_ferramentas_ferramentas,4.44
4,flores,4.42
5,livros_importados,4.40
6,livros_tecnicos,4.37
7,malas_acessorios,4.32
8,alimentos_bebidas,4.32
9,portateis_casa_forno_e_cafe,4.30


### Insight

Product categories with the highest average review scores indicate strong customer satisfaction and potentially higher product quality.

## Business Question 9: Monthly Revenue Trend

**Question:** How did revenue change over time by month?

In [13]:
query = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    ROUND(SUM(oi.price), 2) AS monthly_revenue
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY order_month
ORDER BY order_month;
"""

run_query(query)

,order_month,monthly_revenue
0,2016-09,267.36
1,2016-10,49507.66
2,2016-12,10.90
3,2017-01,120312.87
4,2017-02,247303.02
5,2017-03,374344.30
6,2017-04,359927.23
7,2017-05,506071.14
8,2017-06,433038.60
9,2017-07,498031.48


### Insight

The monthly revenue trend helps identify business growth patterns, seasonality, and changes in sales performance over time.

## Business Question 10: Monthly Order Volume Trend

**Question:** How did order volume change over time by month?

In [14]:
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS order_month,
    COUNT(DISTINCT order_id) AS total_orders
FROM orders
GROUP BY order_month
ORDER BY order_month;
"""

run_query(query)

,order_month,total_orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


### Insight

Monthly order volume shows changes in customer demand over time and can support forecasting and planning.

## Business Question 11: Average Delivery Time by State

**Question:** Which states have the longest average delivery times?

In [15]:
query = """
SELECT
    c.customer_state,
    ROUND(AVG(
        julianday(o.order_delivered_customer_date) 
        - julianday(o.order_purchase_timestamp)
    ), 2) AS avg_delivery_days
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC;
"""

run_query(query)

,customer_state,avg_delivery_days
0,RR,29.39
1,AP,27.19
2,AM,26.43
3,AL,24.54
4,PA,23.77
5,MA,21.57
6,SE,21.52
7,CE,21.27
8,AC,21.04
9,PB,20.43


### Insight

States with longer average delivery times may require logistics improvements to increase customer satisfaction.

## Business Question 12: Top 10 Cities by Number of Orders

**Question:** Which cities generated the highest number of orders?

In [16]:
query = """
SELECT
    c.customer_city,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_city
ORDER BY total_orders DESC
LIMIT 10;
"""

run_query(query)

,customer_city,total_orders
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


### Insight

Cities with the highest order volume are important customer hubs and may be strong targets for marketing and delivery optimization.

## Business Question 13: Top 10 Cities by Revenue

**Question:** Which cities generated the highest revenue?

In [17]:
query = """
SELECT
    c.customer_city,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_city
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query)

,customer_city,total_revenue
0,sao paulo,1914924.54
1,rio de janeiro,992538.86
2,belo horizonte,355611.13
3,brasilia,301920.25
4,curitiba,211738.06
5,porto alegre,190562.08
6,campinas,187844.53
7,salvador,181104.42
8,guarulhos,144268.39
9,niteroi,117907.12


### Insight

Cities with the highest revenue are valuable markets where Olist may focus growth strategies.

## Business Question 14: Top 10 Customers by Spending

**Question:** Which customers spent the most?

In [18]:
query = """
SELECT
    c.customer_unique_id,
    ROUND(SUM(oi.price), 2) AS total_spent
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
ORDER BY total_spent DESC
LIMIT 10;
"""

run_query(query)

,customer_unique_id,total_spent
0,0a0a92112bd4c708ca5fde585afaa872,13440.0
1,da122df9eeddfedc1dc1f5349a1a690c,7388.0
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.0
3,dc4802a71eae9be1dd28f5d788ceb526,6735.0
4,459bef486812aa25204be022145caa62,6729.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,6499.0
6,4007669dec559734d6f53e029e360987,5934.6
7,eebb5dda148d3893cdaf5b5ca3040ccb,4690.0
8,5d0a2980b292d049061542014e8960bf,4599.9
9,48e1ac109decbb87765a3eade6854098,4590.0


### Insight

Top-spending customers represent high-value customers who may be important for loyalty and retention strategies.

## Business Question 15: Order Status Distribution

**Question:** What is the distribution of order statuses?

In [19]:
query = """
SELECT
    order_status,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
"""

run_query(query)

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


### Insight

Order status distribution helps assess operational performance and identify potential issues such as cancellations or unavailable orders.

# Conclusion

This SQL analysis explored key business questions using the Olist E-commerce dataset.

## Key Findings

- A small number of sellers generated a large share of total revenue.
- Some product categories consistently outperformed others in revenue.
- São Paulo and other major states represented the largest markets by both orders and revenue.
- Credit card was the dominant payment method.
- Most customer review scores were positive, suggesting generally strong customer satisfaction.
- Delivery time varied by state, indicating opportunities for logistics improvement.

## Skills Demonstrated

- SQL joins
- SQL aggregation
- Grouping and sorting
- Date-based analysis
- Business insight generation
- SQL analysis using Jupyter Notebook

## Close Database Connection

In [20]:
conn.close()